In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTENC, SMOTEN
from imblearn.combine import SMOTETomek
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer

from sklearn.metrics import fbeta_score
from sklearn.model_selection import GridSearchCV, train_test_split,RandomizedSearchCV
from sklearn.metrics import roc_curve, auc, f1_score, confusion_matrix, roc_auc_score, r2_score, mean_squared_error,precision_recall_curve
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

from sklearn.model_selection import train_test_split


In [33]:
import sys
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install xgboost --upgrade --no-cache-dir


Found existing installation: xgboost 3.2.0
Uninstalling xgboost-3.2.0:
  Successfully uninstalled xgboost-3.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 230.5 MB/s eta 0:00:00a 0:00:01


In [34]:

!{sys.executable} -m pip uninstall -y scikit-learn
!{sys.executable} -m pip install scikit-learn --upgrade --no-cache-dir

Found existing installation: scikit-learn 1.8.0
Uninstalling scikit-learn-1.8.0:
  Successfully uninstalled scikit-learn-1.8.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 176.9 MB/s eta 0:00:00a 0:00:01


In [35]:
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials


In [36]:
import sklearn

print(xgb.__version__)
print(sklearn.__version__)

3.2.0
1.4.2


In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/bank-additional-full.csv',sep=';')
df.head()

df_train,df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['y'])
df_train,df_val = train_test_split(df_train, test_size=0.25, random_state=42, stratify=df_train['y'])
df_train.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
7302,53,retired,married,basic.4y,no,no,no,telephone,may,thu,...,23,999,0,nonexistent,1.1,93.994,-36.4,4.860,5191.0,no
6693,55,retired,divorced,professional.course,no,no,yes,telephone,may,wed,...,2,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
40376,27,student,single,university.degree,no,yes,no,telephone,aug,wed,...,1,0,5,success,-1.7,94.027,-38.3,0.900,4991.6,no
24387,33,technician,single,university.degree,no,no,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no
33177,27,blue-collar,married,basic.9y,no,yes,no,cellular,may,tue,...,1,11,1,success,-1.8,92.893,-46.2,1.291,5099.1,no


In [39]:
df_train['y'] = df_train['y'].map({'yes': 1, 'no': 0}).astype(int)
df_val['y'] = df_val['y'].map({'yes': 1, 'no': 0}).astype(int)
df_test['y'] = df_test['y'].map({'yes': 1, 'no': 0}).astype(int)


In [40]:
X_train = df_train.drop(columns=["y"])
X_val = df_val.drop(columns=["y"])
X_test = df_test.drop(columns=["y"])


categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
X_train[categorical_cols] = X_train[categorical_cols].astype("category")
X_val[categorical_cols] = X_val[categorical_cols].astype("category")
X_test[categorical_cols] = X_test[categorical_cols].astype("category")

y_train = df_train["y"]
y_val = df_val["y"]
y_test = df_test["y"]

In [41]:
def objective(params):
    clf = xgb.XGBClassifier(
    n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        min_child_weight=params['min_child_weight'],  # Мінімальна сума ваг всіх вибірок, необхідна в кінцевому вузлі
        subsample=params['subsample'],  # Частка вибірок, що використовуються для побудови кожного дерева
        colsample_bytree=params['colsample_bytree'],  # Частка ознак, що використовуються при побудові кожного дерева
        gamma=params['gamma'],  # Мінімальне зменшення втрат, необхідне для виконання поділу
        reg_alpha=params['reg_alpha'],  # Параметр регуляризації L1 (Lasso)
        reg_lambda=params['reg_lambda'],  # Параметр регуляризації L2 (Ridge)
        enable_categorical=True,
        missing=np.nan,
        device='cuda'
    )

    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False)
    probs = clf.predict_proba(X_val)[:,1]
    pred = (probs > 0.2).astype(int)  
    f2 = fbeta_score(y_val, pred, beta=2)

    return {'loss': -f2, 'status': STATUS_OK}

In [42]:
space = {
    'n_estimators': hp.quniform('n_estimators', 50,500, 25),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'gamma': hp.uniform('gamma', 0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 0, 1),
}

# Оптимізація
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=20, trials=trials)

# Перетворення значень гіперпараметрів у кінцеві типи
best['n_estimators'] = int(best['n_estimators'])
best['max_depth'] = int(best['max_depth'])
best['min_child_weight'] = int(best['min_child_weight'])

print("Найкращі гіперпараметри: ", best)

100%|██████████| 20/20 [00:25<00:00,  1.27s/trial, best loss: -0.7676426426426426]
Найкращі гіперпараметри:  {'colsample_bytree': np.float64(0.7911464149323284), 'gamma': np.float64(0.4884836836352402), 'learning_rate': np.float64(0.079476072118618), 'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 225, 'reg_alpha': np.float64(0.2249633695432277), 'reg_lambda': np.float64(0.9719296239892484), 'subsample': np.float64(0.5066972217660992)}


In [43]:
final_clf = xgb.XGBClassifier(
    n_estimators=best['n_estimators'],
    learning_rate=best['learning_rate'],
    max_depth=best['max_depth'],
    min_child_weight=best['min_child_weight'],
    subsample=best['subsample'],
    colsample_bytree=best['colsample_bytree'],
    gamma=best['gamma'],
    reg_alpha=best['reg_alpha'],
    reg_lambda=best['reg_lambda'],
    enable_categorical=True,
    missing=np.nan,
    device='cuda'
)
final_clf.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.7911464149323284), device='cuda',
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=np.float64(0.4884836836352402), grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=np.float64(0.079476072118618), max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=4, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=225, n_jobs=None,
              num_parallel_tree=None, ...)

In [44]:
def find_best_threshold_f2(y_true, probas, beta=2):
    """
    Find the best threshold to maximize F-beta score.

    Parameters
    ----------
    y_true : array-like
        True labels (0/1).
    probas : array-like
        Predicted probabilities for positive class.
    beta : float
        Beta parameter for F-beta score (default 2).

    Returns
    -------
    best_t : float
        Best threshold.
    best_fbeta : float
        F-beta score at best threshold.
    best_recall : float
        Recall at best threshold.
    best_precision : float
        Precision at best threshold.
    """
    import numpy as np
    from sklearn.metrics import recall_score, precision_score

    thresholds = np.linspace(0.1, 0.9, 50)
    best_t = thresholds[0]
    best_fbeta = 0
    best_recall = 0
    best_precision = 0

    for t in thresholds:
        preds = (probas >= t).astype(int)
        fbeta = fbeta_score(y_true, preds, beta=beta)
        recall_t = recall_score(y_true, preds)
        precision_t = precision_score(y_true, preds)

        if fbeta > best_fbeta:
            best_fbeta = fbeta
            best_t = t
            best_recall = recall_t
            best_precision = precision_t

    return best_t, best_fbeta, best_recall, best_precision


In [45]:
# ===== TRAIN =====

prob_train = final_clf.predict_proba(X_train)[:,1]
pred_train_thresh = (prob_train >= best_t).astype(int)  # використання твого threshold

precision_train = precision_score(y_train, pred_train_thresh)
recall_train = recall_score(y_train, pred_train_thresh)
f2_train = fbeta_score(y_train, pred_train_thresh, beta=2)

# ===== VAL =====
prob_val = final_clf.predict_proba(X_val)[:,1]
pred_val_thresh = (prob_val >= best_t).astype(int)

precision_val = precision_score(y_val, pred_val_thresh)
recall_val = recall_score(y_val, pred_val_thresh)
f2_val = fbeta_score(y_val, pred_val_thresh, beta=2)

results = {
    'F2_train': f"Train:{round(f2_train,2)} Val:{round(f2_val,2)}",
    'Precision': f"Train:{round(precision_train,2)} Val:{round(precision_val,2)}",
    'Recall': f"Train:{round(recall_train,2)} Val:{round(recall_val,2)}",
    
    'Diff_F2': round(f2_train - f2_val, 2),
    'Diff_Precision': round(precision_train - precision_val, 2),
    'Diff_Recall': round(recall_train - recall_val, 2),
}

results

{'F2_train': 'Train:0.78 Val:0.77',
 'Precision': 'Train:0.47 Val:0.46',
 'Recall': 'Train:0.94 Val:0.93',
 'Diff_F2': np.float64(0.01),
 'Diff_Precision': np.float64(0.01),
 'Diff_Recall': np.float64(0.01)}

In [46]:
prob_val = final_clf.predict_proba(X_val)[:,1]
best_t, best_f2, best_recall, best_precision = find_best_threshold_f2(y_val, prob_val, beta=2)
print("Best threshold:", best_t)
print("F2-score at this threshold:", best_f2)
print("Recall at this threshold:", best_recall)
print("Precision at this threshold:", best_precision)

Best threshold: 0.1653061224489796
F2-score at this threshold: 0.7752293577981652
Recall at this threshold: 0.9105603448275862
Precision at this threshold: 0.4861910241657077


In [47]:

# ===== TEST =====
prob_test = final_clf.predict_proba(X_test)[:,1]
pred_test_thresh = (prob_test >= 0.15).astype(int)

precision_test = precision_score(y_test, pred_test_thresh)
recall_test = recall_score(y_test, pred_test_thresh)
f2_test = fbeta_score(y_test, pred_test_thresh, beta=2)


results = {
    'F2_test': f"TEST:{round(f2_test,2)}",
    'Precision': f"TEST:{round(precision_test,2)}",
    'Recall': f"TEST:{round(recall_test,2)}",
    
}

results

{'F2_test': 'TEST:0.77', 'Precision': 'TEST:0.48', 'Recall': 'TEST:0.92'}